In [1]:
# !pip install shap

In [2]:
import shap

import os
import pickle
import random
import pandas as pd
import copy
import sys
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import time

# plotting
import matplotlib.pyplot as plt

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"]="6"

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
seed = 316
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Load Data

In [6]:
with open('../data/user_sequences.pkl', 'rb') as f:
    user_sequences = pickle.load(f)
with open('../data/train_sequences.pkl', 'rb') as f:
    train_sequences = pickle.load(f)
with open('../data/test_sequences.pkl', 'rb') as f:
    test_sequences = pickle.load(f)


# train_sequences and test_sequences are dictionaries like user_sequences
# Keys: UserIDs (ints)
# Values: list of tuples, each tuple is (movieID, rating)

with open('../data/user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
# with open('../data/idx2user.pkl', 'rb') as f:
#     idx2user = pickle.load(f)
# with open('../data/movie2idx.pkl', 'rb') as f:
#     movie2idx = pickle.load(f)
with open('../data/idx2movie.pkl', 'rb') as f:
    idx2movie = pickle.load(f)


# with open('../data/user_sequences_idx.pkl', 'rb') as f:
#     user_sequences_idx = pickle.load(f)
# with open('../data/train_sequences_idx.pkl', 'rb') as f:
#     train_sequences_idx = pickle.load(f)
# with open('../data/test_sequences_idx.pkl', 'rb') as f:
#     test_sequences_idx = pickle.load(f)

user_embeddings = np.load('../data/user_embeddings.npy')
movie_embeddings = np.load('../data/movie_embeddings.npy')

print(f"User embeddings: {user_embeddings.shape} users)") # should be 6040
print(f"Movie embeddings: {movie_embeddings.shape} movies)") # should be 3706

# pmf_raw_data = pd.read_csv("../data/movielens_1M_pmf_raw_data.csv")
# print(f"Loaded {len(pmf_raw_data)} ratings")

# Load the ratings dictionary
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)

# Dictionary that maps MovieID to Title for Quick Lookup
with open('../data/movie_id_to_title.pkl', 'rb') as f:
    movie_id_to_title = pickle.load(f)

User embeddings: (6040, 100) (should match 6040 users)
Movie embeddings: (3706, 100) (should match 3706 movies)
Loaded 1000209 ratings
Loaded 1000209 entries into ratings_dict


In [7]:
# File paths
DATA_DIR = "../data/ml-1m"
RATINGS_FILE = os.path.join(DATA_DIR, "ratings.dat")
USERS_FILE = os.path.join(DATA_DIR, "users.dat")
MOVIES_FILE = os.path.join(DATA_DIR, "movies.dat")

# Column names
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
users_cols = ['user_id', 'gender', 'age', 'occupation', 'zip_code']
movies_cols = ['movie_id', 'title', 'genres']

# Load data
ratings = pd.read_csv(RATINGS_FILE, sep='::', engine='python', names=ratings_cols, encoding='latin-1')
users = pd.read_csv(USERS_FILE, sep='::', engine='python', names=users_cols, encoding='latin-1')
movies = pd.read_csv(MOVIES_FILE, sep='::', engine='python', names=movies_cols, encoding='latin-1')

In [8]:
class Actor(nn.Module):
    """
    Randomly initialize the Actor πθ
    Thus assuming standard implementation.
    
    maps: state -> action
    From paper: "By two ReLU layers and one Tanh layer, the state representation s 
    is transformed into an action a = π_θ(s)"
    """
    def __init__(self, state_dim, action_dim=100, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        
        # Initialize weights
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)
        
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        # Tanh maps action to [-1, 1]
        action = torch.tanh(self.fc3(x))
        return action

In [10]:
# Load Actor model
version_number = 2

state_dim = 300
actor = Actor(state_dim)
actor = actor.to(device)
# Load the weights into that architecture
actor.load_state_dict(torch.load(f'../models/drr_actor_weights_{version_number}.pth'))
actor.eval()

/tmp/ipykernel_4057545/1457042201.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  actor.load_state_dict(torch.load(f'../models/drr_actor_weights_{version_number}.pth'))


Actor(
  (fc1): Linear(in_features=300, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=100, bias=True)
)

# Algo 2

In [11]:
def get_state(user_idx, history_indices, user_embeddings, movie_embeddings, item_weights=None):
    """
    DRR-ave state for evaluation: s = [u, u⊗g, g]
    
    Args:
        user_idx: index of user
        history_indices: list of item indices (length n_history)
        user_embeddings: array of shape (num_users, embed_dim)
        movie_embeddings: array of shape (num_movies, embed_dim)
        item_weights: optional learned weights (if None, use uniform weights)
    
    Returns:
        state: concatenated array of shape (3*embed_dim,)
    """
    # Get user embedding
    u = user_embeddings[user_idx]  # (embed_dim,)
    
    # Get item embeddings for history
    item_embs = movie_embeddings[history_indices]  # (n_history, embed_dim)
    
    # Apply weights if provided (for evaluation with trained weights)
    if item_weights is not None:
        # Softmax to match training
        weights = np.exp(item_weights) / np.sum(np.exp(item_weights))
        weighted_embs = item_embs * weights[:, np.newaxis]
        g = np.sum(weighted_embs, axis=0)
    else:
        # Simple average (fallback)
        g = np.mean(item_embs, axis=0)
    
    # Element-wise product
    u_g_interaction = u * g
    
    # Concatenate
    state = np.concatenate([u, u_g_interaction, g])
    
    return state.astype(np.float32)

# Evaluation

In [14]:
# Load the saved trajectory
with open('../results/baseline_session_trajectories.pkl', 'rb') as f:
    sessions = pickle.load(f)

In [15]:
def predict_from_history_mask(masks, user_idx, history_list, target_idx):
    scores = []
    for mask in masks:
        selected = [history_list[i] for i in range(len(mask)) if mask[i] > 0.5]
        if len(selected) == 0:
            selected = [history_list[0]]
        
        state = get_state(user_idx, selected, user_embeddings, movie_embeddings)
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        with torch.no_grad():
            action = actor(state_tensor)
            target_emb = torch.FloatTensor(movie_embeddings[target_idx]).to(device)
            score = torch.dot(action.squeeze(), target_emb)
        scores.append(score.item())
    return np.array(scores)

In [16]:
def explain_saved_step(user_id, step_data):
    user_idx = user2idx[user_id]
    history_at_time = step_data['history_at_time']
    target_idx = step_data['recommended_movie_idx']
    
    from functools import partial
    pred_func = partial(predict_from_history_mask, 
                       user_idx=user_idx, 
                       history_list=history_at_time, 
                       target_idx=target_idx)
    
    background_masks = np.random.randint(0, 2, size=(50, len(history_at_time)))
    explainer = shap.KernelExplainer(pred_func, background_masks)
    shap_values = explainer.shap_values(np.ones(len(history_at_time)))
    
    return shap_values

In [17]:
def clean_movie_title(text):
    # 1. Split by '=' and take the first part
    step1 = text.split('=')[0]
    # 2. Split that result by '>' and take the first part
    step2 = step1.split('>')[0]
    
    final_title = step2.strip()
    return final_title

In [18]:
def get_influence_label(weight):
    # Check if the weight is a positive number
    if weight > 0:
        return "+"
    else:
        # This handles both negative numbers AND exactly 0
        return "-"

In [19]:
# Manually chose a few users to explain
users_to_explain = [1, 3, 40, 67, 316]

for user_id in users_to_explain:
    if user_id in sessions:
        session = sessions[user_id]
        print(f"\n{'='*60}")
        print(f"USER {user_id}")
        for step in session['steps']:
            print(f"{step['step']}: {step['recommended_movie_title']} (rating {step['rating']})")
            
        
        for step in session['steps']:
            print(f"\nStep {step['step']}: Recommended {step['recommended_movie_title']} (rating {step['rating']})")
            
            # Run SHAP on this step
            shap_vals = explain_saved_step(user_id, step)
        
            # Get CURRENT history data (includes movies added in previous steps)
            history_at_time = step['history_at_time']
            history_titles = [movie_id_to_title[idx2movie[idx]] for idx in history_at_time]
            
            # Create title-to-rating mapping for CURRENT history
            title_to_rating = {}
            for title, idx in zip(history_titles, history_at_time):
                cleaned = clean_movie_title(title)
                rating = ratings_dict.get((user2idx[user_id], idx), 'N/A')
                title_to_rating[cleaned] = rating
            
            # Create history_influence list
            history_influence = []
            for i, hist_idx in enumerate(history_at_time):
                hist_title = clean_movie_title(movie_id_to_title[idx2movie[hist_idx]])
                history_influence.append((hist_title, shap_vals[i]))
            
            # Convert to DataFrame
            df = pd.DataFrame(history_influence, columns=["Movie in History", "Weight"])
            df["Rating"] = df["Movie in History"].map(title_to_rating)
            df["Influence"] = df["Weight"].apply(get_influence_label)
            df["Weight"] = df["Weight"].round(2)
            df = df.sort_values("Weight", ascending=False)
            
            print(df[["Movie in History", "Rating", "Weight", "Influence"]].to_string())


USER 1
1: Tarzan (1999) (rating 3)
2: Hercules (1997) (rating 4)
3: Toy Story (1995) (rating 5)
4: Pocahontas (1995) (rating 5)
5: Beauty and the Beast (1991) (rating 5)
6: Hunchback of Notre Dame, The (1996) (rating 4)
7: Bug's Life, A (1998) (rating 5)
8: Mulan (1998) (rating 4)
9: Close Shave, A (1995) (rating 3)
10: Aladdin (1992) (rating 4)

Step 1: Recommended Tarzan (1999) (rating 3)
                         Movie in History  Rating  Weight Influence
3                          Ponette (1996)     4.0    0.72         +
1  Snow White and the Seven Dwarfs (1937)     4.0    0.02         +
4                 Schindler's List (1993)     5.0   -0.06         -
0                            Dumbo (1941)     5.0   -0.11         -
2           Miracle on 34th Street (1947)     4.0   -0.29         -

Step 2: Recommended Hercules (1997) (rating 4)
                         Movie in History  Rating  Weight Influence
3                          Ponette (1996)     4.0    0.41         +
0            